# Project 4 — Telecom Customer Churn Analysis
**Level:** Intermediate  |  **Intern:** Sohan Nandi (CSN-OC-2026-474)  |  **Domain:** Data Analysis  |  **October 2026**

## Objective & Business Questions
Analyze customer churn from the Telco dataset (single-file version).

- What is the overall churn rate?
- How does churn vary by contract type, tenure, payment method, internet service?
- How do monthly and total charges differ between churned and retained customers?
- Which customer segments have higher observed churn rates?
- What retention opportunities are suggested by observed patterns?

**Scope note:** The internship task list describes a three-file dataset (Customers, Usage, Billing). The supplied file is a single customer-level table. Analysis uses only columns present; no usage or billing tables are available. This limitation is documented throughout.

In [1]:
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def resolve_repo_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").is_dir() and (parent / "projects").is_dir():
            return parent
        if (parent / "Dataset").is_dir():
            return parent
    return Path(__file__).resolve().parents[3]

REPO_ROOT = resolve_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
def fig_path(name):
    return str(OUT_DIR / name)

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUT_DIR)

# --- Load source data ---
SRC_FILE = DATASET_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
print("Source file:", SRC_FILE)
assert SRC_FILE.exists(), f"Source file missing: {SRC_FILE}"
df = pd.read_csv(SRC_FILE, encoding="utf-8")
print("Loaded shape:", df.shape)


Repo root: C:\Users\sohan\OneDrive\Desktop\CyberNova
Dataset dir: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset
Output dir: outputs
Source file: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset\WA_Fn-UseC_-Telco-Customer-Churn.csv
Loaded shape: (7043, 21)


In [2]:
print("dimensions:", df.shape)
print("columns:", list(df.columns))
print("dtypes:"); print(df.dtypes.to_string())
print("first 3 rows:"); print(df.head(3).to_string())

dimensions: (7043, 21)
columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
first 3 rows:
   customerID  gender  SeniorCitiz

In [3]:
print("null counts:"); print(df.isnull().sum().to_string())
print("duplicate rows:", int(df.duplicated().sum()))
print("customerID unique:", df["customerID"].nunique(), "| rows:", len(df))
print("Churn distribution:", df["Churn"].value_counts().to_dict())
print("Churn rate: %.2f%%" % (df["Churn"]=="Yes").mean()*100)

null counts:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
duplicate rows: 0
customerID unique: 7043 | rows: 7043
Churn distribution: {'No': 5174, 'Yes': 1869}
Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27%Churn rate: 0.27

In [4]:
# Data-quality audit
print("--- data quality audit ---")
print("SeniorCitizen unique:", df["SeniorCitizen"].unique())
print("tenure range:", df["tenure"].min(), "-", df["tenure"].max())
print("MonthlyCharges range:", df["MonthlyCharges"].min(), "-", df["MonthlyCharges"].max())
print("TotalCharges dtype:", df["TotalCharges"].dtype)
# TotalCharges may contain blank/whitespace strings
tc = df["TotalCharges"]
non_num = tc[~tc.astype(str).str.strip().apply(lambda x: x.replace(".","",1).isdigit() if x else False)]
print("TotalCharges non-numeric count:", len(non_num))
if len(non_num):
    print("  samples:", non_num.head(5).tolist())
print("Contract:", df["Contract"].value_counts().to_dict())
print("InternetService:", df["InternetService"].value_counts().to_dict())
print("PaymentMethod:", df["PaymentMethod"].value_counts().to_dict())

--- data quality audit ---
SeniorCitizen unique: [0 1]
tenure range: 0 - 72
MonthlyCharges range: 18.25 - 118.75
TotalCharges dtype: str
TotalCharges non-numeric count: 11
  samples: [' ', ' ', ' ', ' ', ' ']
Contract: {'Month-to-month': 3875, 'Two year': 1695, 'One year': 1473}
InternetService: {'Fiber optic': 3096, 'DSL': 2421, 'No': 1526}
PaymentMethod: {'Electronic check': 2365, 'Mailed check': 1612, 'Bank transfer (automatic)': 1544, 'Credit card (automatic)': 1522}


In [5]:
# Cleaning & validation
clean = df.copy()
# Convert TotalCharges safely: whitespace -> NaN
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce")
print("TotalCharges NaN after conversion:", clean["TotalCharges"].isna().sum())
# Fill NaN TotalCharges with 0 for tenure=0 (new customers), or drop?
# Here we keep NaN and note it; drop for aggregate stats.
clean["tenure_band"] = pd.cut(
    clean["tenure"],
    bins=[0,12,24,36,48,72,100],
    labels=["0-12","12-24","24-36","36-48","48-72","72-100"],
    right=False
)
print("cleaned rows:", len(clean))
clean.to_csv(OUT_DIR / "cleaned_telecom_churn.csv", index=False)
print("saved -> outputs/cleaned_telecom_churn.csv")

TotalCharges NaN after conversion: 11
cleaned rows: 7043
saved -> outputs/cleaned_telecom_churn.csv


In [6]:
# Exploratory analysis
print("--- churn by contract ---")
print(clean.groupby("Contract")["Churn"].value_counts(normalize=True).unstack(fill_value=0).round(4).to_string())
print("--- churn by InternetService ---")
print(clean.groupby("InternetService")["Churn"].value_counts(normalize=True).unstack(fill_value=0).round(4).to_string())
print("--- churn by PaymentMethod ---")
print(clean.groupby("PaymentMethod")["Churn"].value_counts(normalize=True).unstack(fill_value=0).round(4).to_string())
print("--- churn by tenure_band ---")
print(clean.groupby("tenure_band")["Churn"].value_counts(normalize=True).unstack(fill_value=0).round(4).to_string())
print("--- avg charges by churn ---")
print(clean.groupby("Churn")[["MonthlyCharges","TotalCharges"]].mean().round(2).to_string())
print("--- segment: InternetService + Contract ---")
for svc in clean["InternetService"].unique():
    for ctr in clean["Contract"].unique():
        sub = clean[(clean["InternetService"]==svc) & (clean["Contract"]==ctr)]
        if len(sub)==0: continue
        ch = sub["Churn"].value_counts(normalize=True).to_dict()
        print(f"  {svc} / {ctr}: n={len(sub)}, churn={ch.get('Yes',0):.1%}")

--- churn by contract ---
Churn               No     Yes
Contract                      
Month-to-month  0.5729  0.4271
One year        0.8873  0.1127
Two year        0.9717  0.0283
--- churn by InternetService ---
Churn                No     Yes
InternetService                
DSL              0.8104  0.1896
Fiber optic      0.5811  0.4189
No               0.9260  0.0740
--- churn by PaymentMethod ---
Churn                          No     Yes
PaymentMethod                            
Bank transfer (automatic)  0.8329  0.1671
Credit card (automatic)    0.8476  0.1524
Electronic check           0.5471  0.4529
Mailed check               0.8089  0.1911
--- churn by tenure_band ---
Churn            No     Yes
tenure_band                
0-12         0.5172  0.4828
12-24        0.7049  0.2951
24-36        0.7797  0.2203
36-48        0.8048  0.1952
48-72        0.8887  0.1113
72-100       0.9834  0.0166
--- avg charges by churn ---
       MonthlyCharges  TotalCharges
Churn                    

In [7]:
# Charts
import os
os.makedirs(OUT_DIR, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# churn rate by contract
cr_ctr = clean.groupby("Contract")["Churn"].apply(lambda s: (s=="Yes").mean())
cr_ctr.sort_values().plot(kind="barh", ax=axes[0,0], color="#d62728")
axes[0,0].set_title("Churn Rate by Contract"); axes[0,0].set_xlabel("Churn Rate")

# churn rate by tenure band
cr_ten = clean.groupby("tenure_band")["Churn"].apply(lambda s: (s=="Yes").mean())
cr_ten.plot(kind="bar", ax=axes[0,1], color="#ff7f0e")
axes[0,1].set_title("Churn Rate by Tenure Band"); axes[0,1].tick_params(axis="x", rotation=0); axes[0,1].set_ylabel("Churn Rate")

# churn rate by internet service
cr_net = clean.groupby("InternetService")["Churn"].apply(lambda s: (s=="Yes").mean())
cr_net.sort_values().plot(kind="barh", ax=axes[0,2], color="#1f77b4")
axes[0,2].set_title("Churn Rate by Internet Service"); axes[0,2].set_xlabel("Churn Rate")

# MonthlyCharges distribution by churn
for churn_val, col in [("Yes", "#d62728"), ("No", "#2ca02c")]:
    subset = clean.loc[clean["Churn"]==churn_val, "MonthlyCharges"]
    subset.plot(kind="hist", bins=20, ax=axes[1,0], alpha=0.5, color=col, label=churn_val, edgecolor="white")
axes[1,0].set_title("MonthlyCharges by Churn Status"); axes[1,0].set_xlabel("MonthlyCharges ($)"); axes[1,0].legend()

# TotalCharges by churn
for churn_val, col in [("Yes", "#d62728"), ("No", "#2ca02c")]:
    subset = clean.loc[clean["Churn"]==churn_val, "TotalCharges"].dropna()
    subset.plot(kind="hist", bins=20, ax=axes[1,1], alpha=0.5, color=col, label=churn_val, edgecolor="white")
axes[1,1].set_title("TotalCharges by Churn Status"); axes[1,1].set_xlabel("TotalCharges ($)"); axes[1,1].legend()

# segment churn rates (internet + contract)
seg = clean.groupby(["InternetService","Contract"]).apply(
    lambda x: (x["Churn"]=="Yes").mean() if len(x)>0 else np.nan
).dropna().sort_values()
seg.plot(kind="barh", ax=axes[1,2], color="#9467bd")
axes[1,2].set_title("Churn Rate: InternetService × Contract"); axes[1,2].set_xlabel("Churn Rate")

fig.tight_layout()
fig.savefig(fig_path("charts_telecom.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved outputs/charts_telecom.png")

saved outputs/charts_telecom.png


In [8]:
print("KEY FINDINGS (telecom churn):")
print("1. Overall churn rate: %.2f%% (%d of %d customers)" % (
    (clean["Churn"]=="Yes").mean()*100, (clean["Churn"]=="Yes").sum(), len(clean)))
print("2. Contract: Month-to-month churn %.1f%% vs One year %.1f%% vs Two year %.1f%%" % (
    clean.loc[clean.Contract=="Month-to-month","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.Contract=="One year","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.Contract=="Two year","Churn"].apply(lambda x: x=="Yes").mean()*100))
print("3. InternetService: Fiber optic churn %.1f%% vs DSL %.1f%% vs No %.1f%%" % (
    clean.loc[clean.InternetService=="Fiber optic","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.InternetService=="DSL","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.InternetService=="No","Churn"].apply(lambda x: x=="Yes").mean()*100))
print("4. PaymentMethod: Electronic check churn %.1f%% (highest); Credit card (auto) %.1f%% (lowest)" % (
    clean.loc[clean.PaymentMethod=="Electronic check","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.PaymentMethod=="Credit card (automatic)","Churn"].apply(lambda x: x=="Yes").mean()*100))
print("5. Tenure: 0-12 months churn %.1f%%; 72-100 months churn %.1f%%" % (
    clean.loc[clean.tenure_band=="0-12","Churn"].apply(lambda x: x=="Yes").mean()*100,
    clean.loc[clean.tenure_band=="72-100","Churn"].apply(lambda x: x=="Yes").mean()*100))
print("6. Avg MonthlyCharges: churned $%.2f vs retained $%.2f; Avg TotalCharges: churned $%.2f vs retained $%.2f" % (
    clean.loc[clean.Churn=="Yes","MonthlyCharges"].mean(),
    clean.loc[clean.Churn=="No","MonthlyCharges"].mean(),
    clean.loc[clean.Churn=="Yes","TotalCharges"].dropna().mean(),
    clean.loc[clean.Churn=="No","TotalCharges"].dropna().mean()))
print("RECOMMENDATIONS:")
print("- Prioritize retention offers for Month-to-month, Fiber optic, and Electronic check customers.")
print("- High churn in first year: design onboarding/loyalty program targeting tenure 0-12.")
print("- Investigate why Electronic check users churn more (billing friction?).")
print("- Data limitation: single-file; no usage logs or detailed billing history available. "
      "Request multi-file dataset if deeper analysis is needed.")

KEY FINDINGS (telecom churn):
1. Overall churn rate: 26.54% (1869 of 7043 customers)
2. Contract: Month-to-month churn 42.7% vs One year 11.3% vs Two year 2.8%
3. InternetService: Fiber optic churn 41.9% vs DSL 19.0% vs No 7.4%
4. PaymentMethod: Electronic check churn 45.3% (highest); Credit card (auto) 15.2% (lowest)
5. Tenure: 0-12 months churn 48.3%; 72-100 months churn 1.7%
6. Avg MonthlyCharges: churned $74.44 vs retained $61.27; Avg TotalCharges: churned $1531.80 vs retained $2555.34
RECOMMENDATIONS:
- Prioritize retention offers for Month-to-month, Fiber optic, and Electronic check customers.
- High churn in first year: design onboarding/loyalty program targeting tenure 0-12.
- Investigate why Electronic check users churn more (billing friction?).
- Data limitation: single-file; no usage logs or detailed billing history available. Request multi-file dataset if deeper analysis is needed.


In [9]:
print("LIMITATIONS:")
print("- Single customer-level CSV; no separate usage or billing tables as task list described.")
print("- Observational: correlations shown, not causal drivers of churn.")
print("- TotalCharges had %d non-numeric (whitespace) rows; converted to NaN and excluded from mean." % clean["TotalCharges"].isna().sum())
print("- No predictive model trained; segment rates are descriptive only.")
print("CONCLUSION: Churn is concentrated in month-to-month contracts, fiber optic users, electronic check payers, "
      "and early-tenure customers. Targeted retention in these segments is the clearest opportunity.")

LIMITATIONS:
- Single customer-level CSV; no separate usage or billing tables as task list described.
- Observational: correlations shown, not causal drivers of churn.
- TotalCharges had 11 non-numeric (whitespace) rows; converted to NaN and excluded from mean.
- No predictive model trained; segment rates are descriptive only.
CONCLUSION: Churn is concentrated in month-to-month contracts, fiber optic users, electronic check payers, and early-tenure customers. Targeted retention in these segments is the clearest opportunity.
